# Datathon26 — datasets & evaluators demo

Models don't run locally at the datathon: you submit studies to an external
**reportbench** service and score its pre-computed outputs with `radharmony.datathon26`.

- **Classification** — score the `.npz` `global` embedding with a probe.
- **Segmentation** — fold the `.npz` `patches` (+ `grid`) into a dense map; a
  frozen-feature probe scores Dice/IoU against your masks.
- **Report generation** — score the returned `report.txt` vs the reference (RadEval).

Evaluators subclass the standard ones, so k-fold, bootstrap, `macro_average`, the
full RadEval suite, and `generations.csv` all still work.

> **§1** *submits* a run with the one-shot factories; **§2** *scores finished runs*
> with the results datasets.

## The workflow

```
batch prep              reportbench service         radharmony.datathon26
----------              -------------------         ---------------------
sample + anonymize  --> submit studies          --> point a dataset at the
6 CXR datasets          (embed / report task)       results dir + batch df
                                                            |
dataset_{DS1..DS6}.csv      writes per image / study:          v
(labels/reports/masks)    <study>/<model>_<stem>.npz  evaluate() -> metrics
                          <study>/<model>_report.txt
```

You supply the **batch table** (labels / reports / masks) and the **results dir**;
a dataset joins them and an evaluator scores.

## 0. Setup

In [2]:
import pandas as pd

import radharmony.datathon26  # noqa: F401  -> registers all datathon26_* keys
from radharmony.datathon26 import (
    EmbeddingResultsDataset,
    ReportResultsDataset,
    PatchSegResultsDataset,
    DatathonLinearProbeEvaluator,
    DatathonReportGenerationEvaluator,
    DatathonLinearProbeSegEvaluator,
    viewwise_embedding_path,
    perstudy_report_path,
)
from radharmony.evaluator.metrics.language import LIGHT_METRICS
from radharmony.registry import list_datasets
from radharmony.evaluator._registry import list_evaluators

print("datasets: ", sorted(k for k in list_datasets() if "datathon" in k))
print("evaluators:", sorted(k for k in list_evaluators() if "datathon" in k))

datasets:  ['datathon26_embedding', 'datathon26_report', 'datathon26_segmentation']
evaluators: ['datathon26_conv_probe_seg', 'datathon26_knn_probe', 'datathon26_linear_probe', 'datathon26_linear_probe_seg', 'datathon26_prototype_probe', 'datathon26_report_generation', 'datathon26_svm_probe', 'datathon26_upernet_seg']


## 1. One-shot: submit DS6 → classification + segmentation

Dataset **DS6 (SIIM-ACR-PTX)** is the worked test: 10,000 frontal CXRs, a binary
`pneumothorax` label, and — uniquely among the six — **real segmentation masks**
(`mask_path`). It has no finished run yet, so we submit it. The **one-shot
factories** stage the harmonized frame + raw images, submit the `embed` task (via
`ReportBenchClient` → `rbclient.py`), wait, and hand back a ready dataset.

Fill in your `INSTRUCTIONS.md` values and run **on the datathon server** — it submits
real jobs. Template, not "Run All"-safe.

> - **Cls + seg share one embed job:** `DatathonEmbeddingDataset` and
>   `DatathonPatchEmbeddingDataset` submit the same `embed` task (`.npz` carry both
>   `global` and `patches`) — submit once, reuse with `skip_submit=True` + same `run_name`.
> - **No reports:** SIIM ships no findings text, so report generation lives in §2b (on DS1).
> - **Staging:** `prepare` copies the images once into the run folder.

In [ ]:
from radharmony.datathon26 import (
    ReportBenchClient,
    DatathonEmbeddingDataset,
)
import os

# Files you prepare under DATA_DIR are owned by you, but the zeus service reads
# them as a different user — so they must be world-readable or `submit` 500s.
# umask 0o022 makes the staged + prepared files o+r. One-time, in a terminal:
#   chmod o+x <DATA_DIR>   (lets the service traverse into the batch root)
os.umask(0o022)

# --- fill these in (see your team INSTRUCTIONS.md), then run on the server ---
RBCLIENT_PATH = "/mnt/NAS4/projects/bkhosra/sharing/datathon/skill/rbclient.py"
DATA_DIR = "/mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test"
API_KEY = "rb_..."
MODEL = "model-a"

# os.makedirs(DATA_DIR, exist_ok=True)

# Dataset DS6 (SIIM-ACR-PTX): frontal CXRs + pneumothorax label + real masks.
DS6_DIR = "/mnt/NAS4/projects/fli40_sharing/datathon26_cxr_datasets/"
IMAGES_DIR = f"{DS6_DIR}/images"  # image_path in the frame is relative to this
df_DS6 = pd.read_csv(
    f"{DS6_DIR}/dataset_DS6.csv"
)  # .head(200)  # small demo subset (drop .head for all)

client = ReportBenchClient(
    rbclient_path=RBCLIENT_PATH, data_dir=DATA_DIR, api_key=API_KEY
)

# Classification — submit the embed task once as run_DS6; the .npz `global` vector
# feeds the probe. The label column (pneumothorax) is auto-detected from the frame.
# Fresh run_name avoids colliding with an earlier prepared batch (the client
# won't overwrite an existing run).
emb_ds = DatathonEmbeddingDataset(
    df_DS6,
    IMAGES_DIR,
    client=client,
    model=MODEL,
    run_name="run_DS6_demo",
    skip_submit=False,
)
DatathonLinearProbeEvaluator(dataset=emb_ds, n_folds=5).evaluate()

[reportbench] staged 10000 studies into /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/.staging/run_DS6_demo
[reportbench] $ python3 /mnt/NAS4/projects/bkhosra/sharing/datathon/skill/rbclient.py config --api-key rb_***REDACTED*** --data-dir /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test
saved /home/fli40/.config/rbclient/config.json
{
  "base_url": "http://zeus.hiti.emory.edu:8077",
  "api_key": "rb_7Wq\u2026",
  "data_dir": "/mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test"
}
(config file: /home/fli40/.config/rbclient/config.json)
[reportbench] $ python3 /mnt/NAS4/projects/bkhosra/sharing/datathon/skill/rbclient.py prepare --from /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/.staging/run_DS6_demo --name run_DS6_demo
prepared 10000 studies in /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/run_DS6_demo
  optional: add ONE manifest.json in this batch folde

extracting embeddings:   0%|          | 0/157 [00:00<?, ?batch/s]

,n_train,fold,seed,bootstrap,label,auroc,auprc,f1,accuracy,balanced_accuracy,tpr,tnr,ppv,npv,mcc,threshold
0,-1,0,-1,-1,pneumothorax,0.962462,0.873003,0.792523,0.8890,0.914169,0.959276,0.869063,0.675159,0.986880,0.740536,0.429853
1,-1,1,-1,-1,pneumothorax,0.962544,0.883797,0.810127,0.9025,0.904538,0.908297,0.900778,0.731107,0.970650,0.753508,0.558429
2,-1,2,-1,-1,pneumothorax,0.950447,0.851359,0.754435,0.8685,0.893470,0.937355,0.849586,0.631250,0.980147,0.693638,0.378273
3,-1,3,-1,-1,pneumothorax,0.959578,0.860332,0.798010,0.8985,0.908486,0.926097,0.890874,0.701049,0.977591,0.744600,0.483844
4,-1,4,-1,-1,pneumothorax,0.952718,0.851073,0.761194,0.8720,0.890293,0.923077,0.857510,0.647619,0.975182,0.697245,0.479927
5,-1,0,-1,-1,macro_average,0.962462,0.873003,0.792523,0.8890,0.914169,0.959276,0.869063,0.675159,0.986880,0.740536,0.429853
6,-1,1,-1,-1,macro_average,0.962544,0.883797,0.810127,0.9025,0.904538,0.908297,0.900778,0.731107,0.970650,0.753508,0.558429
7,-1,2,-1,-1,macro_average,0.950447,0.851359,0.754435,0.8685,0.893470,0.937355,0.849586,0.631250,0.980147,0.693638,0.378273
8,-1,3,-1,-1,macro_average,0.959578,0.860332,0.798010,0.8985,0.908486,0.926097,0.890874,0.701049,0.977591,0.744600,0.483844
9,-1,4,-1,-1,macro_average,0.952718,0.851073,0.761194,0.8720,0.890293,0.923077,0.857510,0.647619,0.975182,0.697245,0.479927


**Segmentation factory — real masks.** `DatathonPatchEmbeddingDataset` reuses the
same `embed` job (`run_DS6` above, so `skip_submit=True`), folds each `.npz`'s
`patches` → `[dim, H, W]` via its `grid`, and pairs it with DS6's real `mask_path`.
One embed job feeds both classification and segmentation.

In [6]:
from radharmony.datathon26 import DatathonPatchEmbeddingDataset

# Segmentation on DS6's REAL masks. mask_path in dataset_DS6.csv is relative
# (masks_DS6/<study>/frontal.png); mask_dir resolves it (masks live in their own
# tree, not under the embed outputs). Reuses the embed run_DS6_demo submitted above
# (skip_submit=True); drop skip_submit to submit a fresh run.
seg_ds = DatathonPatchEmbeddingDataset(
    df_DS6,
    IMAGES_DIR,
    client=client,
    model=MODEL,
    run_name="run_DS6_demo",
    skip_submit=True,  # reuse the embed run from the classification cell
    mask_col="mask_path",
    mask_dir=DS6_DIR,  # root for the relative mask paths
    mask_size=224,  # probe output resolution; masks resized to match
)
DatathonLinearProbeSegEvaluator(
    dataset=seg_ds,
    num_classes=2,
    n_folds=5,
    device="cpu",
    autocast_dtype=None,
).evaluate().groupby("label")[["dice", "iou"]].mean().round(4)

Exception ignored in: Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c42e6b1d120><function _MultiProcessingDataLoaderIter.__del__ at 0x7c42e6b1d120><function _MultiProcessingDataLoaderIter.__del__ at 0x7c42e6b1d120>


Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
            self._shutdown_workers()Exception ignored in: self._shutdown_workers()self._shutdown_workers()
<function _MultiProcessingDataLoaderIter.__del__ at 0x7c42e6b1d120>

  File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dat

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

,dice,iou
label,,
class_0,0.9986,0.9971
class_1,0.2179,0.1525
macro_average,0.2179,0.1525


## 2. Score more datasets — read back or submit

- **§2a** — read back the run **you** submitted in §1 (dataset **DS6**) with no further
  service call: its `.npz` are already on disk, scored for classification + segmentation.
- **§2b** — submit **dataset DS1** yourself (the same way as §1): it carries reference
  reports + 14 finding labels (DS6 has neither), so it drives report generation and a
  richer multi-label cls.

A results dataset points at a finished `_reportbench_out/<model>/` and scores it. The
service writes, per image / per study:

```
DS1_patient_00018_study_00001/
    model-a_pa.npz        # per IMAGE: `global` [dim] + `patches` [n, dim] + `grid`
    model-a_lateral.npz
    model-a_report.txt    # per STUDY
```

Match granularity with the `result_path=` resolver: embeddings/patches per-image
(`viewwise_embedding_path`), reports per-study (`perstudy_report_path`).

Registry keys: `datathon26_{embedding,segmentation,report}`,
`datathon26_{linear,knn,svm,prototype}_probe`, `datathon26_linear_probe_seg`
(+ conv/upernet), `datathon26_report_generation`.

### 2a. Read back the run you submitted (DS6)

§1 submitted DS6's `embed` job; its `.npz` are on disk now. The **results datasets**
read them with no service call — `EmbeddingResultsDataset` takes the `global` vector
(classification), `PatchSegResultsDataset` folds `patches` + `grid` into a dense map
and pairs it with DS6's real masks (segmentation). Needs §1 to have run.
(`DATA_DIR`, `MODEL`, `df_DS6`, `DS6_DIR` come from §1.)

In [7]:
# Classification, read-back: point EmbeddingResultsDataset at DS6's finished embed run
# (submitted in §1) and score the `global` vectors — no service call. One row per
# image; viewwise_embedding_path maps each to <study>/<model>_<stem>.npz.
DS6_OUT_DIR = (
    f"{DATA_DIR}/run_DS6_demo/_reportbench_out/{MODEL}"  # <study>/model-a_frontal.npz
)

ds6_cls_ds = EmbeddingResultsDataset(
    DS6_OUT_DIR,
    harmonized_df=df_DS6,
    result_path=viewwise_embedding_path(MODEL),
)
DatathonLinearProbeEvaluator(
    dataset=ds6_cls_ds, n_folds=5, device="cpu", autocast_dtype=None, num_workers=2
).evaluate().groupby("label")[["auroc", "auprc", "f1"]].mean().round(4)

extracting embeddings:   0%|          | 0/157 [00:00<?, ?batch/s]

,auroc,auprc,f1
label,,,
macro_average,0.9575,0.8639,0.7833
pneumothorax,0.9575,0.8639,0.7833


In [8]:
# Segmentation, read-back: same DS6 run, folds each .npz's patches -> [dim, H, W] via
# its grid and pairs with DS6's real masks (relative mask_path + mask_dir). Nothing
# re-runs.
seg_ds = PatchSegResultsDataset(
    DS6_OUT_DIR,
    harmonized_df=df_DS6,  # real mask_path already present (relative)
    result_path=viewwise_embedding_path(MODEL),
    mask_dir=DS6_DIR,  # root for the relative mask paths
    mask_size=224,
)
DatathonLinearProbeSegEvaluator(
    dataset=seg_ds,
    num_classes=2,
    n_folds=5,
    device="cpu",
    autocast_dtype=None,
    num_workers=2,
).evaluate().groupby("label")[["dice", "iou"]].mean().round(4)

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c42e6b1d120>
Traceback (most recent call last):
  File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7c42e6b1d120>  
Traceback (most recent call last):
   File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
      self._shutdown_workers()^
^^  File "/home/fli40/RadHarmony/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():^
^  ^ ^ ^ ^ ^ 
^^  File "/home/fli40/.local/share/uv/python/cpython-3.12.9-linux-x86_64-gnu/lib/python3.12/multiprocessing/process.py"

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

training:   0%|          | 0/20 [00:00<?, ?epoch/s]

,dice,iou
label,,
class_0,0.9985,0.9970
class_1,0.2151,0.1504
macro_average,0.2151,0.1504


### 2b. Submit DS1 yourself — reports + multi-label cls

Rather than read a run someone else submitted, submit **DS1 (MIMIC)** yourself, exactly
like §1. Because the run then covers *precisely* your frame, there's no `manifest.json`
filtering — every result file lines up 1:1 with the CSV, so the results dataset builds
without dropping rows. DS1 carries reference reports + 14 finding labels, so it drives
**report generation** (DS6 has none) and a richer **multi-label** classification.

Reuses `client`, `IMAGES_DIR`, and `MODEL` from §1. Classification uses the `embed`
task; report generation uses the `report` task — each factory call prepares + submits
its **own** run (`prepare` wipes + recreates the run folder, so two tasks can't share
one). DS1 is large; `.head()` a subset for a quick demo.

> **Regenerating a run.** Re-running with a `run_name` that already exists fails (`prepare` won't overwrite: *"… already exists (use --force …)"*). To regenerate in place, delete the run folder first — `rm -rf <DATA_DIR>/<run_name>` — then re-run with the same name; or just pick a fresh `run_name`.

In [15]:
# Submit DS1's embed task yourself (like §1) and score the `global` vectors — no
# organizer corpus, no manifest filter. Reuses client / IMAGES_DIR / MODEL from §1.
DS1_CSV = "/mnt/NAS4/projects/fli40_sharing/datathon26_cxr_datasets/dataset_DS1.csv"
df_DS1 = pd.read_csv(DS1_CSV)  # .head(200)  # small demo subset (drop .head for all)
df_DS1 = df_DS1[df_DS1["view_position"] == "AP"].head(200)

cls_ds = DatathonEmbeddingDataset(
    df_DS1,
    IMAGES_DIR,
    client=client,
    model=MODEL,
    run_name="run_DS1_embed",
    skip_submit=False,  # set True to reuse an already-submitted run_DS1_embed
)
# DS1's 14 labels -> macro-average across findings, unlike DS6's single binary label.
cls_results = DatathonLinearProbeEvaluator(
    dataset=cls_ds, n_folds=5, device="cpu", autocast_dtype=None, num_workers=2
).evaluate()
cls_results.groupby("label")[["auroc", "auprc", "f1"]].mean().round(4)

[reportbench] staged 200 studies into /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/.staging/run_DS1_embed
[reportbench] $ python3 /mnt/NAS4/projects/bkhosra/sharing/datathon/skill/rbclient.py prepare --from /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/.staging/run_DS1_embed --name run_DS1_embed
prepared 200 studies in /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/run_DS1_embed
  optional: add ONE manifest.json in this batch folder, keyed by study, to label views and give each study an indication:
    {"studies": {"<study_folder>": {"images": [{"path": "img", "view": "ap|pa|lateral|ll"}], "indication": "..."}}}
next:  rbclient.py submit --model model-a --task report --input run_DS1_embed
[reportbench] $ python3 /mnt/NAS4/projects/bkhosra/sharing/datathon/skill/rbclient.py submit --model model-a --task embed --input run_DS1_embed
submitted 7c7adf171b9d4ad29b884ba551e18f63
  model=model-a task=embed stud

extracting embeddings:   0%|          | 0/4 [00:00<?, ?batch/s]

,auroc,auprc,f1
label,,,
atelectasis,0.7065,0.3988,0.5046
cardiomegaly,0.7885,0.5526,0.6504
consolidation,0.6711,0.1435,0.1600
edema,0.9216,0.7307,0.6885
enlarged_cardiomediastinum,0.7569,0.2661,0.5000
fracture,0.7895,0.1857,0.2538
lung_lesion,NaN,NaN,NaN
lung_opacity,0.7653,0.4853,0.5464
macro_average,0.8312,0.5877,0.6212


In [ ]:
from radharmony.datathon26 import DatathonReportDataset

# Report generation — submit the SAME frame for the `report` task in its OWN run
# (prepare wipes the folder, so it can't reuse run_DS1_embed). The model writes
# <study>/<model>_report.txt; the frame's `report` column is the reference, and the
# factory dedupes to one row per study.
rep_ds = DatathonReportDataset(
    df_DS1,
    IMAGES_DIR,
    client=client,
    model=MODEL,
    run_name="run_DS1_report2",
    skip_submit=False,  # set True to reuse an already-submitted run_DS1_report2
)
DatathonReportGenerationEvaluator(
    dataset=rep_ds, metrics=LIGHT_METRICS, ref_section="findings", num_workers=2
).evaluate()

[reportbench] staged 200 studies into /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/.staging/run_DS1_report2
[reportbench] $ python3 /mnt/NAS4/projects/bkhosra/sharing/datathon/skill/rbclient.py prepare --from /mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/data_DS6_test/.staging/run_DS1_report2 --name run_DS1_report2
